# Agente Vitrinifarne — Notebook 2: índice e agente

Aqui o agente ganha vida. Ao final deste notebook você faz uma pergunta em português e
recebe uma resposta baseada nos documentos, com a fonte citada.

**O caminho:**

1. Ler os documentos (reaproveitando o módulo `src/leitores.py`)
2. Quebrar os textos em pedaços menores — os *chunks*
3. Transformar cada pedaço em vetor numérico — o *embedding*
4. Guardar os vetores num índice de busca — o *FAISS*
5. A cada pergunta: buscar os pedaços mais parecidos e mandar para o Gemini responder

**Pré-requisito:** chave de API do Google Gemini salva nos Secrets do Colab com o nome
`GOOGLE_API_KEY`.

---

## 1. Instalar as bibliotecas

Além das do notebook anterior, entram três novas:

| Biblioteca | Para quê |
|---|---|
| `langchain-text-splitters` | quebrar o texto em pedaços com sobreposição |
| `langchain-google-genai` | conversar com o Gemini (embeddings e respostas) |
| `faiss-cpu` | guardar os vetores e buscar por similaridade |

Leva cerca de um minuto.

In [1]:
!pip install -q pypdf python-docx python-pptx openpyxl beautifulsoup4 pandas
!pip install -q langchain-text-splitters langchain-google-genai faiss-cpu
print("Pronto.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 37.6 MB/s eta 0:00:00
Pronto.


## 2. A chave de API

**Nunca escreva a chave dentro do código.** Se você fizer isso e subir o notebook para o
GitHub, a chave fica pública — bots varrem o GitHub procurando exatamente isso, e o Google
revoga a chave em minutos.

O Colab tem um cofre para isso. No ícone de **chave** (🔑) na barra lateral esquerda:

1. Clique em *Adicionar novo secret*
2. Nome: `GOOGLE_API_KEY`
3. Valor: cole a chave que você gerou no Google AI Studio
4. Ative o botão *Acesso ao notebook*

A célula abaixo lê do cofre. A chave nunca aparece no código nem no resultado.

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

chave = os.environ["GOOGLE_API_KEY"]
print(f"Chave carregada: {chave[:6]}...{chave[-4:]} ({len(chave)} caracteres)")

Chave carregada: AQ.Ab8...pU3g (53 caracteres)


## 3. Descobrir qual modelo usar

Os nomes dos modelos do Gemini mudam com o tempo. Em vez de fixar um nome que pode estar
desatualizado, a célula abaixo testa alguns candidatos e fica com o primeiro que responder.

Se nenhum funcionar, o erro exibido já diz o motivo — geralmente chave inválida ou cota
esgotada.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODELO = "gemini-flash-latest"

modelo_teste = ChatGoogleGenerativeAI(model=MODELO, temperature=0)
print("Modelo:", MODELO, "→", modelo_teste.invoke("responda apenas: ok").content)

Modelo: gemini-flash-latest → [{'type': 'text', 'text': 'ok', 'extras': {'signature': 'EtQCCtECARFNMg+84LC/hMcsU45ApSYcOXLCG+PIIldN9wJ/QzKmfTkJL+cb0m283Z0ZnuTxbE4sUWPFJa9bRKWSZiIoj8PSQtLLC7yW0y7qUvRuxJjBSaT3SC/HrRxUk2eYh75WfcXLodpMc2JUypM/I0X/spdx9Ucrgd6213BX+6YHIhJ9DHs6N3IRels3FMARTqW1PJfqWAbzlh++DpGR4m9Gs7I4wB9iEJbCHFru3hiGk6u53YKvv+kPm+ptj4bzXE6g34+2K2K1R33GF3SXSjrpAVfsnIwqhVgXl8J3WCTAAAKF7v/h0Q3kOKGb1+HCMpmllO4FXQschgTMy3gOgPTiOZcqVfoRtZrtUT7kwlx+qoh3n362KSlVuqq0QyjqcVYJBvrvrJZhif4B2ZZIP6A3PxgUy79M2bB1CJZn8iZVT2RlTFjNK88dIcAPpqdrIEohXg=='}}]


In [7]:
import os, requests

chave = os.environ.get("GOOGLE_API_KEY", "")
print("Tamanho da chave:", len(chave))
print("Tem espaço ou quebra de linha sobrando?", chave != chave.strip())

chave = chave.strip()
os.environ["GOOGLE_API_KEY"] = chave

resposta = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models",
    params={"key": chave},
)
print("Status:", resposta.status_code, "\n")

if resposta.status_code == 200:
    print("Modelos disponíveis para esta chave:")
    for m in resposta.json().get("models", []):
        if "generateContent" in m.get("supportedGenerationMethods", []):
            print(" -", m["name"].replace("models/", ""))
else:
    print(resposta.text[:1500])

Tamanho da chave: 53
Tem espaço ou quebra de linha sobrando? False
Status: 200 

Modelos disponíveis para esta chave:
 - gemini-2.5-flash
 - gemini-2.5-pro
 - gemini-2.0-flash
 - gemini-2.0-flash-001
 - gemini-2.0-flash-lite-001
 - gemini-2.0-flash-lite
 - gemini-2.5-flash-preview-tts
 - gemini-2.5-pro-preview-tts
 - gemma-4-26b-a4b-it
 - gemma-4-31b-it
 - gemini-flash-latest
 - gemini-flash-lite-latest
 - gemini-pro-latest
 - gemini-2.5-flash-lite
 - gemini-2.5-flash-image
 - gemini-3-pro-preview
 - gemini-3-flash-preview
 - gemini-3.1-pro-preview
 - gemini-3.1-pro-preview-customtools
 - gemini-3.1-flash-lite-preview
 - gemini-3.1-flash-lite
 - gemini-3-pro-image-preview
 - gemini-3-pro-image
 - nano-banana-pro-preview
 - gemini-3.1-flash-image-preview
 - gemini-3.1-flash-image
 - gemini-3.1-flash-lite-image
 - gemini-3.5-flash
 - gemini-3.5-flash-lite
 - gemini-omni-flash-preview
 - gemini-3.6-flash
 - lyria-3-clip-preview
 - lyria-3-pro-preview
 - gemini-3.1-flash-tts-preview
 - gem

In [8]:
import os, requests
from langchain_google_genai import ChatGoogleGenerativeAI
import langchain_google_genai as lgg

print("versão da lib:", lgg.__version__, "\n")

# 1) o erro completo, sem filtro
try:
    ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0).invoke("diga: ok")
    print("FUNCIONOU com gemini-flash-latest")
except Exception as e:
    print("ERRO REAL:\n", str(e)[:1200])

# 2) quais modelos de embedding existem hoje
r = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                 params={"key": os.environ["GOOGLE_API_KEY"]})
print("\nModelos de embedding disponíveis:")
for m in r.json().get("models", []):
    if "embedContent" in m.get("supportedGenerationMethods", []):
        print(" -", m["name"].replace("models/", ""))

versão da lib: 4.3.2 

FUNCIONOU com gemini-flash-latest

Modelos de embedding disponíveis:
 - gemini-embedding-001
 - gemini-embedding-2-preview
 - gemini-embedding-2


## 4. Trazer os documentos e o módulo de leitura

Clonamos o repositório e apontamos o Python para a pasta `src/`, onde mora o
`leitores.py`. É o mesmo código do Notebook 1, agora empacotado como módulo — assim ele
é escrito uma vez e usado em qualquer lugar.

In [16]:
!rm -rf vitrinifarne-agente
!git clone -q https://github.com/francielleneves/vitrinifarne-agente.git

import sys, json
from pathlib import Path

RAIZ = Path("vitrinifarne-agente")
sys.path.append(str(RAIZ / "src"))

from leitores import extrair

PASTA_DOCS = RAIZ / "docs"
manifesto = json.loads((PASTA_DOCS / "manifesto.json").read_text(encoding="utf-8"))

base = []
for doc in manifesto["documentos"]:
    base.append({**doc, "texto": extrair(PASTA_DOCS / doc["arquivo"], doc["formato"])})

print(f"{len(base)} documentos lidos, {sum(len(d['texto']) for d in base)} caracteres.")

8 documentos lidos, 45678 caracteres.


## 5. Quebrar em pedaços

Por que não mandar o documento inteiro para o Gemini? Dois motivos: custo (você paga por
texto enviado) e precisão (o modelo se perde em texto longo e a resposta fica genérica).

Então quebramos tudo em pedaços de mais ou menos 900 caracteres. Dois detalhes importam:

**A sobreposição de 150 caracteres.** Cada pedaço repete o final do anterior. Sem isso,
uma frase pode ser cortada ao meio e a informação se perde nas duas metades. Imagine o
corte caindo entre "o prazo de estorno é de" e "10 dias úteis" — nenhum dos dois pedaços
responderia à pergunta.

**A ordem dos separadores.** O `RecursiveCharacterTextSplitter` tenta cortar primeiro em
quebras de parágrafo, depois em linhas, depois em pontos finais. Só corta no meio de uma
palavra em último caso. Assim os pedaços respeitam a estrutura do texto.

Repare também que cada pedaço recebe um cabeçalho com o nome e a versão do documento.
Isso viaja junto para o Gemini e é o que permite a ele citar a fonte corretamente.

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", " ", ""],
    length_function=len,
)

chunks = []
for doc in base:
    for i, pedaco in enumerate(divisor.split_text(doc["texto"]), start=1):
        chunks.append({
            "texto": f"[{doc['titulo']} — versão {doc['versao']}]\n{pedaco}",
            "doc_id": doc["id"],
            "titulo": doc["titulo"],
            "arquivo": doc["arquivo"],
            "versao": doc["versao"],
            "categoria": doc["categoria"],
            "parte": i,
        })

print(f"{len(chunks)} pedaços gerados.\n")

import pandas as pd
tabela = pd.DataFrame(chunks)
tabela["tamanho"] = tabela["texto"].str.len()
tabela.groupby(["doc_id", "titulo"]).agg(
    pedacos=("parte", "count"), tamanho_medio=("tamanho", "mean")
).round(0)

64 pedaços gerados.



,,pedacos,tamanho_medio
doc_id,titulo,,
DOC-01,Política de Privacidade,9,715.0
DOC-02,Termos e Condições de Uso e de Compra,9,806.0
DOC-03,Perguntas Frequentes,7,794.0
DOC-04,Guia de Envios e Entregas,6,869.0
DOC-05,Tabela de Prazos e Fretes por Região,8,864.0
DOC-06,Catálogo de Produtos,9,790.0
DOC-07,Política de Reembolso e Devoluções,10,889.0
DOC-08,Treinamento de Atendimento ao Cliente,6,674.0


## 6. Transformar texto em números

Este é o conceito central do projeto.

Um *embedding* é uma lista de números que representa o significado de um texto. Textos com
sentido parecido geram listas de números parecidas — mesmo sem compartilhar nenhuma
palavra. É por isso que "quando meu dinheiro volta?" consegue encontrar um trecho que fala
em "prazo de estorno".

O modelo do Google devolve 768 números para cada pedaço. Nós normalizamos esses vetores
para que a comparação entre eles seja o cosseno do ângulo — um valor entre 0 e 1, onde 1
significa "praticamente o mesmo assunto".

Esta célula faz 60 chamadas à API e leva cerca de 30 segundos.

In [18]:
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings

MODELO_EMBEDDING = "models/gemini-embedding-001"
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

textos = [c["texto"] for c in chunks]


def gerar_embeddings(textos, tamanho_lote=10):
    """Gera os vetores em lotes pequenos, com plano B de um por vez."""
    vetores = []
    for i in range(0, len(textos), tamanho_lote):
        lote = textos[i:i + tamanho_lote]
        try:
            vetores.extend(embeddings.embed_documents(lote))
        except Exception:
            for texto in lote:
                vetores.append(embeddings.embed_query(texto))
        print(f"  {min(i + tamanho_lote, len(textos))}/{len(textos)} pedaços")
    return np.array(vetores, dtype="float32")


vetores = gerar_embeddings(textos)

print("\nFormato da matriz:", vetores.shape, "→", vetores.shape[0],
      "pedaços de", vetores.shape[1], "números cada")
print("Primeiros 8 números do pedaço 1:", vetores[0][:8].round(3))

  10/64 pedaços
  20/64 pedaços
  30/64 pedaços
  40/64 pedaços
  50/64 pedaços
  60/64 pedaços
  64/64 pedaços

Formato da matriz: (64, 3072) → 64 pedaços de 3072 números cada
Primeiros 8 números do pedaço 1: [-0.025  0.011  0.022 -0.073  0.011  0.     0.032  0.013]


## 7. Montar o índice de busca

O FAISS guarda os vetores de um jeito que permite achar os mais parecidos rapidamente.

`IndexFlatIP` compara por produto interno. Como normalizamos os vetores antes, esse
produto interno é exatamente a similaridade por cosseno.

Para 60 pedaços isso é instantâneo. A mesma estrutura funcionaria com milhões — é a razão
de usar FAISS em vez de comparar um por um no Python.

In [19]:
import faiss

faiss.normalize_L2(vetores)

indice = faiss.IndexFlatIP(vetores.shape[1])
indice.add(vetores)

print(f"Índice criado com {indice.ntotal} vetores de {indice.d} dimensões.")

Índice criado com 64 vetores de 3072 dimensões.


## 8. A busca

A função abaixo recebe uma pergunta, converte em vetor pelo mesmo modelo e devolve os
pedaços mais parecidos, com a nota de similaridade.

Antes de partir para as respostas, vale rodar e olhar: se a busca traz o trecho errado,
não existe prompt que salve a resposta. **Quando o agente erra, o problema quase sempre
está aqui, não no modelo.**

In [20]:
import re
import unicodedata

PALAVRAS_IGNORADAS = {
    "para", "com", "uma", "quando", "qual", "quais", "onde", "como", "meu", "minha",
    "este", "esta", "esse", "essa", "dos", "das", "que", "nao", "sim", "por", "sobre",
    "pode", "posso", "tem", "quanto", "quantos", "quero", "preciso", "mais", "sua",
}


def normalizar(texto):
    """Deixa tudo minúsculo e sem acento, para comparar palavras."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


# Normaliza uma vez só. Se você refizer os chunks, rode esta linha de novo.
CHUNKS_NORMALIZADOS = [normalizar(c["texto"]) for c in chunks]


def termos_da_pergunta(pergunta):
    palavras = re.findall(r"\w+", normalizar(pergunta))
    return {p for p in palavras if len(p) > 3 and p not in PALAVRAS_IGNORADAS}


def buscar(pergunta, quantidade=6, peso_palavras=0.20):
    """Combina busca por significado (vetores) com busca por palavra (termos da pergunta)."""
    vetor = np.array([embeddings.embed_query(pergunta)], dtype="float32")
    faiss.normalize_L2(vetor)

    quantos = min(len(chunks), max(quantidade * 4, 20))
    notas, posicoes = indice.search(vetor, quantos)
    notas_vetor = {int(p): float(n) for n, p in zip(notas[0], posicoes[0])}

    termos = termos_da_pergunta(pergunta)
    acertos_por_pedaco = {
        i: sum(1 for t in termos if t in texto)
        for i, texto in enumerate(CHUNKS_NORMALIZADOS)
    }

    # Candidatos: o que o vetor achou + qualquer pedaço com 2 ou mais palavras da pergunta
    candidatos = set(notas_vetor)
    candidatos |= {i for i, a in acertos_por_pedaco.items() if a >= 2}

    resultados = []
    for posicao in candidatos:
        pedaco = dict(chunks[posicao])
        similaridade = notas_vetor.get(posicao, 0.0)
        acertos = acertos_por_pedaco[posicao]
        pedaco["similaridade"] = similaridade
        pedaco["palavras"] = acertos
        pedaco["nota_final"] = similaridade + peso_palavras * (acertos / max(len(termos), 1))
        resultados.append(pedaco)

    resultados.sort(key=lambda r: r["nota_final"], reverse=True)
    return resultados[:quantidade]


for r in buscar("Comprei uma estante modular para Belém. Quando chega e o frete é grátis?"):
    print(f"{r['nota_final']:.3f} (vetor {r['similaridade']:.3f}, {r['palavras']} palavras)"
          f"  {r['doc_id']}  {r['titulo']} parte {r['parte']}")

0.782 (vetor 0.724, 2 palavras)  DOC-03  Perguntas Frequentes parte 4
0.773 (vetor 0.716, 2 palavras)  DOC-04  Guia de Envios e Entregas parte 3
0.771 (vetor 0.714, 2 palavras)  DOC-05  Tabela de Prazos e Fretes por Região parte 1
0.767 (vetor 0.710, 2 palavras)  DOC-05  Tabela de Prazos e Fretes por Região parte 2
0.755 (vetor 0.698, 2 palavras)  DOC-05  Tabela de Prazos e Fretes por Região parte 3
0.752 (vetor 0.695, 2 palavras)  DOC-03  Perguntas Frequentes parte 3


## 9. O prompt

Agora juntamos tudo: os trechos encontrados vão para o Gemini junto com a pergunta e um
conjunto de instruções.

As instruções não são enfeite. Cada linha resolve um problema real:

- **"Use apenas os trechos"** evita que o modelo invente a partir do que ele aprendeu na
  internet. Sem essa linha, ele responde sobre a política de devolução de outra loja
  qualquer e soa igualmente convincente.
- **"Cite o documento e a versão"** é o que transforma isso numa base de conhecimento
  corporativa, e não num chute bem escrito.
- **"Se não estiver nos trechos, diga que não encontrou"** é a instrução mais importante.
  Um agente que admite não saber é confiável. Um que inventa é pior que não ter agente.

In [21]:
INSTRUCOES = """Você é o assistente interno da Vitrinifarne, uma loja online de casa e decoração.
Sua função é responder perguntas de colaboradores e clientes usando exclusivamente a
documentação oficial da empresa.

Regras:
1. Responda apenas com base nos trechos fornecidos abaixo. Não invente políticas, prazos
   ou valores que não estejam nos trechos. Você pode usar conhecimento geral apenas para
   relacionar termos equivalentes — por exemplo, reconhecer que uma capital pertence a um
   estado citado nos trechos.
2. Sempre cite o documento e a versão de onde tirou a informação.
3. Se a resposta exigir combinar informações de documentos diferentes, faça a combinação
   e cite todas as fontes usadas.
4. Se a informação não estiver nos trechos, responda exatamente: "Não encontrei essa
   informação na documentação disponível." e sugira qual área procurar.
5. Responda em português do Brasil, de forma direta e objetiva. Sem saudações.
6. Valores em reais e prazos devem ser reproduzidos exatamente como aparecem nos trechos.

TRECHOS DA DOCUMENTAÇÃO:
{contexto}

PERGUNTA: {pergunta}

RESPOSTA:"""

modelo = ChatGoogleGenerativeAI(model=MODELO, temperature=0)


def texto_da_resposta(resposta):
    """As versões novas do Gemini devolvem blocos, não texto puro. Aqui juntamos só o texto."""
    conteudo = resposta.content
    if isinstance(conteudo, str):
        return conteudo.strip()
    partes = []
    for bloco in conteudo:
        if isinstance(bloco, str):
            partes.append(bloco)
        elif isinstance(bloco, dict) and bloco.get("type") == "text":
            partes.append(bloco.get("text", ""))
    return "\n".join(p for p in partes if p).strip()


def responder(pergunta, quantidade=5, mostrar_fontes=True):
    """Busca os trechos relevantes e pede ao Gemini uma resposta baseada neles."""
    achados = buscar(pergunta, quantidade)

    contexto = "\n\n---\n\n".join(a["texto"] for a in achados)
    prompt = INSTRUCOES.format(contexto=contexto, pergunta=pergunta)

    resposta = texto_da_resposta(modelo.invoke(prompt))

    if mostrar_fontes:
        fontes = []
        for a in achados:
            marca = f"{a['titulo']} (v{a['versao']})"
            if marca not in fontes:
                fontes.append(marca)
        resposta += "\n\nTrechos consultados: " + "; ".join(fontes)

    return resposta


print("Agente pronto.")

Agente pronto.


## 10. Testar

Cinco perguntas escolhidas para exercitar coisas diferentes.

A quinta é a mais importante: **nenhum documento sozinho responde a ela**. É preciso o
catálogo (para saber que a estante é sob encomenda e quanto custa), a planilha de prazos
(região Norte) e o guia de entregas (o mínimo de frete grátis no Norte é diferente).
Se o agente acertar essa, ele está realmente cruzando fontes.

In [22]:
PERGUNTAS = [
    "Qual o prazo para desistir de uma compra?",
    "Em quantas vezes posso parcelar e qual a parcela mínima?",
    "Qual o prazo de entrega expresso para o interior do Nordeste?",
    "Um cliente quer devolver um produto de higiene pessoal com o lacre aberto. Pode?",
    "Comprei uma estante modular para Belém. Quando chega e o frete é grátis?",
]

for i, pergunta in enumerate(PERGUNTAS, start=1):
    print("=" * 78)
    print(f"PERGUNTA {i}: {pergunta}")
    print("-" * 78)
    print(responder(pergunta))
    print()

PERGUNTA 1: Qual o prazo para desistir de uma compra?
------------------------------------------------------------------------------
O prazo para desistir de uma compra (arrependimento) é de 7 dias corridos contados a partir do recebimento do produto.

Fontes:
- Termos e Condições de Uso e de Compra — versão 1.4
- Política de Reembolso e Devoluções — versão 2.2

Trechos consultados: Termos e Condições de Uso e de Compra (v1.4); Política de Reembolso e Devoluções (v2.2); Perguntas Frequentes (v3.0)

PERGUNTA 2: Em quantas vezes posso parcelar e qual a parcela mínima?
------------------------------------------------------------------------------
É possível parcelar em até 6 vezes sem juros no cartão de crédito, com parcela mínima de R$ 50,00. Acima de 6 parcelas, há incidência de juros de 1,99% ao mês, exibidos/informados antes da confirmação do pedido.

Fontes:
- Perguntas Frequentes — versão 3.0
- Termos e Condições de Uso e de Compra — versão 1.4

Trechos consultados: Perguntas Freque

## 11. Testar o que o agente NÃO sabe

Tão importante quanto acertar é recusar direito. Estas perguntas não têm resposta na
documentação: o agente deve dizer que não encontrou, e não inventar.

Se ele inventar aqui, aumente a clareza da regra 4 nas instruções ou reduza a quantidade
de trechos buscados.

In [23]:
FORA_DO_ESCOPO = [
    "Qual o faturamento da empresa no último trimestre?",
    "Vocês entregam em Portugal?",
    "Quem é o CEO da Vitrinifarne?",
]

for pergunta in FORA_DO_ESCOPO:
    print("-" * 78)
    print("PERGUNTA:", pergunta)
    print(responder(pergunta, mostrar_fontes=False))
    print()

------------------------------------------------------------------------------
PERGUNTA: Qual o faturamento da empresa no último trimestre?
Não encontrei essa informação na documentação disponível. Sugiro procurar a área Financeira ou de Controladoria.

------------------------------------------------------------------------------
PERGUNTA: Vocês entregam em Portugal?
Não encontrei essa informação na documentação disponível. Sugiro consultar a área de Logística.

------------------------------------------------------------------------------
PERGUNTA: Quem é o CEO da Vitrinifarne?
Não encontrei essa informação na documentação disponível. Sugiro procurar a área de Atendimento ao Cliente (SAC) ou a Diretoria.



## 12. Salvar o índice

Gerar os embeddings custa tempo e cota de API. Salvando o índice, o notebook seguinte
(e o servidor na OCI) carrega tudo pronto, sem refazer as chamadas.

São dois arquivos: o `.faiss` com os vetores e o `.json` com os textos e metadados
correspondentes. Os dois precisam andar juntos — a posição no índice é o que liga um
ao outro.

In [24]:
faiss.write_index(indice, "indice.faiss")

with open("chunks.json", "w", encoding="utf-8") as arquivo:
    json.dump(chunks, arquivo, ensure_ascii=False, indent=2)

for nome in ["indice.faiss", "chunks.json"]:
    print(f"{nome}: {Path(nome).stat().st_size / 1024:.1f} KB")

print("\nBaixe os dois pelo painel de arquivos à esquerda.")

indice.faiss: 768.0 KB
chunks.json: 65.5 KB

Baixe os dois pelo painel de arquivos à esquerda.


---

## O que este notebook fez

Todo o diagrama da arquitetura, do começo ao fim:

| Etapa | Célula |
|---|---|
| Ler os 8 formatos | 4 |
| Quebrar em pedaços | 5 |
| Gerar embeddings | 6 |
| Montar o índice | 7 |
| Buscar por similaridade | 8 |
| Responder com o Gemini | 9 |

## Próximos passos

1. **Interface** — uma tela simples com Gradio, para não demonstrar o projeto no Colab
2. **Implantação na OCI** — subir a aplicação e deixar acessível por um link
3. **README** — arquitetura, exemplos de perguntas e respostas, instruções de execução